# L2 · Salida estructurada en profundidad: de un mensaje de cliente a datos

*Advanced Topics in Analytics II - Generative AI* · Instructor: Sergio A. Mora Pardo

En [L1](L01-IntroToLLMs.ipynb) pedimos una salida estructurada sencilla: una respuesta y su justificación. En el soporte de **ACME** llegan cientos de mensajes al día y cada uno debe convertirse en un registro: categoría, cliente, montos, urgencia. Hoy diseñamos ese esquema con cuidado: campos opcionales, categorías cerradas, listas, validaciones y qué hacer cuando el modelo no devuelve lo esperado.

## Objetivos

Al terminar podrás:

1. Diseñar un esquema de Pydantic con descripciones que guían al modelo.
2. Usar `Literal` para categorías cerradas y `None` para datos ausentes.
3. Extraer listas de objetos anidados, como los cargos de un reclamo.
4. Validar reglas de negocio con Pydantic y ver qué pasa cuando fallan.
5. Elegir el método de salida estructurada y leer la respuesta cruda con `include_raw`.

## Contenido

- [0. Preparación del modelo](#0.-Preparación-del-modelo)
- [1. Sin esquema: texto libre](#1.-Sin-esquema:-texto-libre)
- [2. Un esquema con descripciones](#2.-Un-esquema-con-descripciones)
- [3. Categorías cerradas y datos ausentes](#3.-Categorías-cerradas-y-datos-ausentes)
- [4. Listas de objetos](#4.-Listas-de-objetos)
- [5. Validar reglas de negocio](#5.-Validar-reglas-de-negocio)
- [6. Método y respuesta cruda](#6.-Método-y-respuesta-cruda)
- [7. Varios mensajes con `batch`](#7.-Varios-mensajes-con-batch)
- [Práctica breve](#Práctica-breve)
- [Referencias](#Referencias)

El recorrido realiza **8 llamadas a Claude Haiku**, incluidas las tres entradas de `batch`. La sección 5 (validación) no llama al modelo.

## 0. Preparación del modelo

El mismo modelo de L1.

In [1]:
from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic

# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

model = ChatAnthropic(
    model="claude-haiku-5-5",
    max_tokens=600,
    thinking={"type": "disabled"},
    timeout=60,
    max_retries=2,
)

## 1. Sin esquema: texto libre

Un mensaje real de un cliente: desordenado, con dos cargos y una fecha relativa. Si le pedimos al modelo que «extraiga los datos», contesta con texto: útil para leer, inútil para guardar en una base o enrutar el ticket (**1 llamada**).

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage

customer_message = (
    "Hola, soy el cliente C-1042. Ayer me llegaron dos cobros de mi tarjeta débito ACME, "
    "uno de 185.000 y otro igual, en la misma tienda en línea. Yo solo compré una vez. "
    "Necesito que me devuelvan la plata porque con eso pago el arriendo."
)

response = model.invoke([
    SystemMessage("Extrae los datos importantes del mensaje del cliente."),
    HumanMessage(customer_message),
])
print(response.text)

Datos extraídos del mensaje:

- **Identificación del cliente:** C-1042
- **Tipo de tarjeta:** Débito ACME
- **Motivo del contacto:** Cobro duplicado
- **Monto por cobro:** $185.000
- **Cantidad de cobros:** 2 (idénticos)
- **Fecha:** Ayer
- **Comercio:** Tienda en línea (no identificada)
- **Compras realizadas por el cliente:** 1
- **Solicitud:** Devolución del monto duplicado ($185.000)
- **Urgencia / motivo personal:** Pago de arriendo
- **Monto total potencialmente a devolver:** $185.000


## 2. Un esquema con descripciones

Sin modelo. Un esquema de Pydantic define **qué campos** queremos y **de qué tipo**. Cada `Field(description=...)` es una instrucción para el modelo: dice qué poner en el campo y en qué formato. El *docstring* de la clase también llega al modelo como descripción del objeto.

`model_json_schema()` muestra lo que recibirá Claude: es el mismo JSON Schema que en L4 describía los argumentos de una herramienta.

In [3]:
import json

from pydantic import BaseModel, Field


class TicketBasics(BaseModel):
    """Datos básicos de un mensaje de soporte de ACME."""

    customer: str = Field(description="Código del cliente con el formato C-1234.")
    summary: str = Field(description="Resumen del problema en una frase, en español.")
    amount_cop: int = Field(description="Monto principal reclamado en pesos colombianos, sin separadores.")


print(json.dumps(TicketBasics.model_json_schema(), ensure_ascii=False, indent=2))

{
  "description": "Datos básicos de un mensaje de soporte de ACME.",
  "properties": {
    "customer": {
      "description": "Código del cliente con el formato C-1234.",
      "title": "Customer",
      "type": "string"
    },
    "summary": {
      "description": "Resumen del problema en una frase, en español.",
      "title": "Summary",
      "type": "string"
    },
    "amount_cop": {
      "description": "Monto principal reclamado en pesos colombianos, sin separadores.",
      "title": "Amount Cop",
      "type": "integer"
    }
  },
  "required": [
    "customer",
    "summary",
    "amount_cop"
  ],
  "title": "TicketBasics",
  "type": "object"
}


In [4]:
basics_model = model.with_structured_output(TicketBasics)
basics = basics_model.invoke([HumanMessage(customer_message)])
basics

TicketBasics(customer='C-1042', summary='Cobro duplicado en tarjeta débito ACME: dos cargos de 185.000 COP en la misma tienda en línea por una sola compra; el cliente solicita la devolución del cobro duplicado.', amount_cop=185000)

Ahora `basics.amount_cop` es un entero y `basics.customer` un texto con formato: se pueden guardar, comparar y enrutar. ¿Qué monto puso? El cliente reclama **uno** de los dos cobros de 185.000, pero el esquema no dice si `amount_cop` es el total cobrado o lo que pide que le devuelvan. **Un campo ambiguo produce datos ambiguos**: la descripción es parte del diseño.

## 3. Categorías cerradas y datos ausentes

Dos decisiones de diseño que evitan la mayoría de los errores:

- **`Literal`** limita un campo a valores conocidos. Sin él, el modelo puede escribir «cargo doble», «duplicado» o «cobro repetido», y el enrutamiento falla.
- **`... | None`** permite decir «no está». Si un campo es obligatorio y el dato no aparece, el modelo lo **inventa** para cumplir el esquema. Con `None`, puede dejarlo vacío.

El mensaje dice «ayer», no una fecha: pedimos `charge_date` como fecha exacta o `None` (**1 llamada**).

In [5]:
from typing import Literal


class TicketTriage(BaseModel):
    """Clasificación de un mensaje de soporte de ACME."""

    customer: str = Field(description="Código del cliente con el formato C-1234.")
    category: Literal["cargo_duplicado", "bloqueo_tarjeta", "desembolso", "mora", "otro"] = Field(
        description="Categoría del problema."
    )
    urgency: Literal["alta", "media", "baja"] = Field(
        description="alta si el cliente no puede cubrir un gasto esencial o hay sospecha de fraude."
    )
    refund_requested_cop: int | None = Field(
        description="Monto que el cliente pide que le devuelvan, en pesos; None si no pide devolución."
    )
    charge_date: str | None = Field(
        description="Fecha exacta del cargo en formato AAAA-MM-DD; None si el mensaje no da una fecha exacta."
    )


triage = model.with_structured_output(TicketTriage).invoke([HumanMessage(customer_message)])
print(triage.model_dump_json(indent=2))

{
  "customer": "C-1042",
  "category": "cargo_duplicado",
  "urgency": "alta",
  "refund_requested_cop": 185000,
  "charge_date": null
}


Revisa `charge_date`: el mensaje dice «ayer» y el modelo no sabe qué día es hoy. ¿Dejó `None` o inventó una fecha? Y `refund_requested_cop`: ¿puso 185.000, lo que pide devolver, o 370.000, lo cobrado? La descripción ahora lo dice explícitamente.

## 4. Listas de objetos

Un reclamo puede traer varios cargos. Un campo `list[Charge]` le pide al modelo **un objeto por cargo**, cada uno con sus propios campos. Es la misma idea del `{% for %}` de L3, pero en la salida (**1 llamada**).

In [6]:
class Charge(BaseModel):
    """Un cargo mencionado por el cliente."""

    amount_cop: int = Field(description="Monto del cargo en pesos, sin separadores.")
    merchant: str = Field(description="Comercio del cargo tal como lo describe el cliente.")
    recognized: bool = Field(description="true si el cliente reconoce el cargo como suyo.")


class ChargeClaim(BaseModel):
    """Reclamo por cargos en la tarjeta de un cliente de ACME."""

    customer: str = Field(description="Código del cliente con el formato C-1234.")
    charges: list[Charge] = Field(description="Todos los cargos que menciona el cliente, uno por cargo.")


claim = model.with_structured_output(ChargeClaim).invoke([HumanMessage(customer_message)])
for charge in claim.charges:
    print(charge)

amount_cop=185000 merchant='tienda en línea' recognized=True
amount_cop=185000 merchant='tienda en línea' recognized=False


¿Separó los dos cobros? ¿Marcó uno como reconocido y el otro no? En la salida guardada lo hizo bien, pero al repetirlo a veces marca **ambos** como no reconocidos: con dos cargos idénticos, `recognized` no dice **cuál** es el legítimo, y el modelo tiene que adivinar. Si un campo depende de esa adivinanza, conviene una pregunta que no la necesite, como «¿cuántos cargos reconoce?». Fíjate también en la extracción validada de la sección siguiente: ahí marcó los dos como no reconocidos. Con una lista bien definida, el sistema puede abrir una disputa solo por los cargos no reconocidos.

## 5. Validar reglas de negocio

Sin modelo. El esquema garantiza **tipos**, no que los datos tengan sentido. Pydantic permite añadir reglas con `field_validator`: aquí, que el código del cliente tenga el formato `C-1234` y que ningún monto sea negativo. Las mismas reglas valen si los datos llegan del modelo, de un formulario o de otro sistema.

In [7]:
import re

from pydantic import ValidationError, field_validator


class ValidatedCharge(Charge):
    @field_validator("amount_cop")
    @classmethod
    def positive_amount(cls, value):
        if value <= 0:
            raise ValueError("el monto debe ser positivo")
        return value


class ValidatedClaim(ChargeClaim):
    charges: list[ValidatedCharge] = Field(description="Todos los cargos que menciona el cliente, uno por cargo.")

    @field_validator("customer")
    @classmethod
    def customer_code(cls, value):
        if not re.fullmatch(r"C-\d{4}", value):
            raise ValueError("el código del cliente debe tener el formato C-1234")
        return value


try:
    ValidatedClaim.model_validate(
        {"customer": "1042", "charges": [{"amount_cop": -185000, "merchant": "tienda", "recognized": False}]}
    )
except ValidationError as error:
    print(error)

2 validation errors for ValidatedClaim
customer
  Value error, el código del cliente debe tener el formato C-1234 [type=value_error, input_value='1042', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
charges.0.amount_cop
  Value error, el monto debe ser positivo [type=value_error, input_value=-185000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


El error nombra cada campo que falla y por qué. Si el modelo devuelve algo que no pasa la validación, `with_structured_output` lanza una excepción: es mejor un error visible que un registro con datos imposibles en la base.

## 6. Método y respuesta cruda

`with_structured_output` puede pedirle la estructura a Claude de dos formas:

| Método | Cómo funciona | Garantía |
|---|---|---|
| `function_calling` (por defecto) | El esquema viaja como una herramienta y el modelo «la llama» con los datos | El modelo casi siempre cumple, pero no está garantizado |
| `json_schema` | Usa las salidas estructuradas nativas de Claude | La respuesta **siempre** cumple el esquema |

Con `include_raw=True` recibimos también el mensaje crudo del modelo y un posible error de parseo, en lugar de una excepción. Así podemos registrar los fallos y revisarlos (**1 llamada**).

In [8]:
robust_model = model.with_structured_output(ValidatedClaim, method="json_schema", include_raw=True)
result = robust_model.invoke([HumanMessage(customer_message)])

print("¿Error de parseo?", result["parsing_error"])
print("Tokens de salida:", result["raw"].usage_metadata["output_tokens"])
result["parsed"]

¿Error de parseo? None
Tokens de salida: 92


ValidatedClaim(customer='C-1042', charges=[ValidatedCharge(amount_cop=185000, merchant='tienda en línea (ACME débito)', recognized=False), ValidatedCharge(amount_cop=185000, merchant='tienda en línea (ACME débito)', recognized=False)])

`json_schema` es la opción más segura cuando el esquema es fijo, como en el juez de L11. `function_calling` sigue siendo útil cuando el mismo modelo también debe poder usar otras herramientas.

## 7. Varios mensajes con `batch`

En producción, el mismo extractor procesa muchos mensajes. `batch`, como en L1, envía cada mensaje en una llamada independiente y devuelve los resultados en el mismo orden (**3 llamadas**).

In [9]:
inbox = [
    "Soy C-2210. Firmé el Crédito Express hace una hora y todavía no me llega la plata.",
    "Buenas, C-3307 aquí. Tengo 75 días de mora y me quedé sin trabajo, ¿qué puedo hacer?",
    "Me bloquearon la tarjeta y estoy de viaje, soy el cliente C-4415. ¡Urgente!",
]

triage_model = model.with_structured_output(TicketTriage)
for ticket in triage_model.batch([[HumanMessage(message)] for message in inbox]):
    print(ticket.customer, "·", ticket.category, "·", ticket.urgency)

C-2210 · desembolso · media
C-3307 · mora · alta
C-4415 · bloqueo_tarjeta · alta


Con estos registros, un sistema puede enrutar cada ticket a su equipo: es lo que hará el grafo de **routing** de la semana 8, con un nodo por categoría.

## Práctica breve

1. Añade a `TicketTriage` un campo `channel: Literal["app", "correo", "telefono"] | None` y pruébalo con los mensajes de la sección 7. ¿Qué pone cuando el mensaje no dice el canal?
2. Cambia la descripción de `amount_cop` en `TicketBasics` para que no sea ambigua y repite la sección 2.
3. Añade a `ValidatedClaim` una regla que exija al menos un cargo no reconocido. ¿Qué pasa con un mensaje que no reclama nada?
4. Compara `method="function_calling"` y `method="json_schema"` con un mensaje que no tenga código de cliente. ¿Cuál devuelve un error y cuál inventa un código?

## Referencias

**Learning LangChain**, Mayo Oshin y Nuno Campos, O'Reilly, capítulo 1: [`h-structured.py`](https://github.com/langchain-ai/learning-langchain/blob/master/ch1/py/h-structured.py).

- [Salida estructurada en LangChain](https://docs.langchain.com/oss/python/langchain/structured-output)
- [Validadores de Pydantic](https://docs.pydantic.dev/latest/concepts/validators/)